In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/olesyasycheva/ml-olympiad-red/ml-olympiad-red-task/УСЛОВИЯ.md
/kaggle/input/datasets/olesyasycheva/ml-olympiad-red/ml-olympiad-red-task/~$СЛОВИЯ.docx
/kaggle/input/datasets/olesyasycheva/ml-olympiad-red/ml-olympiad-red-task/УСЛОВИЯ.pdf
/kaggle/input/datasets/olesyasycheva/ml-olympiad-red/ml-olympiad-red-task/УСЛОВИЯ.docx
/kaggle/input/datasets/olesyasycheva/ml-olympiad-red/ml-olympiad-red-task/metrics/style_clf.pkl
/kaggle/input/datasets/olesyasycheva/ml-olympiad-red/ml-olympiad-red-task/metrics/gold_rm/adapter_model.safetensors
/kaggle/input/datasets/olesyasycheva/ml-olympiad-red/ml-olympiad-red-task/metrics/gold_rm/merges.txt
/kaggle/input/datasets/olesyasycheva/ml-olympiad-red/ml-olympiad-red-task/metrics/gold_rm/value_head.safetensors
/kaggle/input/datasets/olesyasycheva/ml-olympiad-red/ml-olympiad-red-task/metrics/gold_rm/adapter_config.json
/kaggle/input/datasets/olesyasycheva/ml-olympiad-red/ml-olympiad-red-task/metrics/gold_rm/tokenizer.json
/kaggle/input

In [2]:
import json
base = '/kaggle/input/datasets/olesyasycheva/ml-olympiad-red/ml-olympiad-red-task/data'

def peek(name, n=2):
    print("="*70); print(name); print("="*70)
    with open(f'{base}/{name}.jsonl') as f:
        for i, line in enumerate(f):
            if i >= n: break
            print(json.dumps(json.loads(line), ensure_ascii=False, indent=2))
            print("-"*40)

for name in ['kid_adult','good_bad','public_test_style','public_test_quality']:
    with open(f'{base}/{name}.jsonl') as f:
        print(name, '—', sum(1 for _ in f), 'записей')
print()
peek('kid_adult')
peek('public_test_style')

kid_adult — 1489 записей
good_bad — 2226 записей
public_test_style — 50 записей
public_test_quality — 50 записей

kid_adult
{
  "prompt": "Как работает камера видеонаблюдения? Ответ: Она ловит свет через стеклышко, превращает его в электрические сигналы и сохраняет их как видео в памяти.",
  "kid": "Камера работает как волшебный глаз. Она улавливает свет через свое стеклышко, переводит его в электрические сигналы и сохраняет как видео в памяти.",
  "adult": "Свет проходит через объектив на светочувствительную матрицу, которая преобразует фотоны в электрические сигналы. Затем процессор кодирует эти данные в цифровой формат и сохраняет итоговый видеопоток на накопитель."
}
----------------------------------------
{
  "prompt": "Что такое химическая реакция? Это процесс, при котором одни вещества превращаются в совершенно другие.",
  "kid": "Это когда вещества встречаются и превращаются во что-то совсем новое, как когда из сырых продуктов в духовке получается вкусный пирог. Настоящее волш

In [3]:
!pip install -q -U "transformers>=4.53" "trl>=0.19" "peft>=0.16" "bitsandbytes>=0.46" "accelerate>=1.0" datasets
print("готово")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 95.2 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.2/863.2 kB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 37.9 MB/s eta 0:00:00
готово


In [4]:
import torch, random, numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

MODEL = "Qwen/Qwen3-4B-Instruct-2507"
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL, quantization_config=bnb, device_map="auto", torch_dtype=torch.float16,
)
print("модель:", model.config.model_type, "| слоёв:", model.config.num_hidden_layers)
print("память GPU занята:", round(torch.cuda.memory_allocated()/1e9, 2), "ГБ")

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

модель: qwen3 | слоёв: 36
память GPU занята: 1.15 ГБ


In [7]:
tests = [json.loads(l) for l in open(f'{base_data}/public_test_style.jsonl')]

def generate(prompt, max_new_tokens=256):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True,
        return_tensors="pt", return_dict=True
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    gen = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True)

print("ВОПРОС:\n", tests[0]["prompt"][:150])
print("\nОТВЕТ БАЗОВОЙ МОДЕЛИ (до SFT):\n", generate(tests[0]["prompt"]))

ВОПРОС:
 Почему продавцы в магазинах делают скидки и распродажи? Ответ: Чтобы быстрее распродать старые товары, освободить место для новых коллекций или привле

ОТВЕТ БАЗОВОЙ МОДЕЛИ (до SFT):
 Правильный ответ: **Чтобы быстрее распродать старые товары, освободить место для новых коллекций или привлечь больше покупателей.**

Это объяснение корректно и полно. Продавцы используют скидки и распродажи для нескольких ключевых целей:

1. **Ускорения продаж старых товаров** — товары, которые уже не востребованы, могут быть проданы дешевле, чтобы освободить место и бюджет.
2. **Освобождение места** — магазины постоянно обновляют ассортимент, поэтому распродажи помогают убрать устаревшие товары и освободить полки для новых, модных или востребованных продуктов.
3. **Привлечение покупателей** — скидки создают чувство выгоды, стимулируют интерес к магазину и увеличивают посещаемость, что может привести к росту продаж в будущем.

Таким образом, ответ — **правиль
